# 环境配置

In [1]:
from semantic_kernel import __version__

__version__

'1.41.0'

In [2]:
import os
from dotenv import load_dotenv
from rich import print

load_dotenv(".env/bailian", override=True)
# load_dotenv(".env/openrouter", override=True)
print(os.getenv("OPENAI_RESPONSES_MODEL_ID"))

qwen3.5-plus

# 01 使用responses接口

国内免费模型只有百炼平台的qwen的部分模型支持/responses，参见[文档](https://bailian.console.aliyun.com/cn-beijing/?tab=api#/api/?type=model&url=3016539)。

openrouter平台支持[Responses接口](https://openrouter.ai/docs/api/reference/responses/overview)，但还处于早期开发阶段。

In [3]:
import os
from semantic_kernel.agents import OpenAIResponsesAgent

In [ ]:
client = OpenAIResponsesAgent.create_client()

agent = OpenAIResponsesAgent(
    ai_model_id=os.getenv("OPENAI_RESPONSES_MODEL_ID"),
    client=client,
    instructions="Answer questions about the world in one sentence.",
    name="Expert",
    # 使用openrouter接口必须禁用store
    # 禁用store后百炼接口也正常了
    store_enabled=False,
)

In [ ]:
user_chat = [
    "Hi, my name is John Doe.",
    "Why is the sky blue?",
    "What is the speed of light?",
    "What is my name?",
]

In [ ]:
for user_input in user_chat:
    print(f"[red]User: {user_input}[/red]")

    response = await agent.get_response(messages=user_input)
    print(f"[green]{response.name}: {response.content}[/green]")


# 02 添加了thread

有上下文感知了

In [4]:
from semantic_kernel.agents import AgentThread

thread: AgentThread = None

In [ ]:
for user_input in user_chat:
    print(f"[red]User: {user_input}[/red]")

    response = await agent.get_response(messages=user_input, thread=thread)
    print(f"[green]{response.name}: {response.content}[/green]")

    thread = response.thread

# 03 使用插件 MenuPlugin

In [5]:
from typing import Annotated
from semantic_kernel.functions import kernel_function

In [ ]:
class MenuPlugin:
    """A sample Menu Plugin used for the concept sample."""

    @kernel_function(description="Provides a list of specials from the menu.")
    def get_specials(self) -> Annotated[str, "Returns the specials from the menu."]:
        return """
        Special Soup: Clam Chowder
        Special Salad: Cobb Salad
        Special Drink: Chai Tea
        """

    @kernel_function(description="Provides the price of the requested menu item.")
    def get_item_price(
        self, menu_item: Annotated[str, "The name of the menu item."]
    ) -> Annotated[str, "Returns the price of the menu item."]:
        return "$9.99"

In [ ]:
client = OpenAIResponsesAgent.create_client()

agent_menu = OpenAIResponsesAgent(
    ai_model_id=os.getenv("OPENAI_RESPONSES_MODEL_ID"),
    client=client,
    instructions="Answer questions about the menu.",
    name="Host",
    plugins=[MenuPlugin()],
    # 使用openrouter接口必须禁用store
    # 禁用store后百炼接口也正常了
    store_enabled=False,
)

In [ ]:
user_chat_menu = [
    "Hello",
    "What is the special soup?",
    "What is the special drink?",
    "How much is it?",
    "Thank you",
]

In [ ]:
thread: AgentThread = None

for user_input in user_chat_menu:
    print(f"[red]User ▶︎ {user_input}[/red]")

    response = await agent_menu.get_response(messages=user_input, thread=thread)
    print(f"[green]{response.name} ▶︎ {response.content}[/green]")

    thread = response.thread


# 04 使用Web Search工具 ✅

In [ ]:
web_search_tool = OpenAIResponsesAgent.configure_web_search_tool()

client = OpenAIResponsesAgent.create_client()

agent_ws = OpenAIResponsesAgent(
    ai_model_id=os.getenv("OPENAI_RESPONSES_MODEL_ID"),
    client=client,
    instructions="Answer questions from the user about performing web searches for news.",
    name="NewsSearcher",
    tools=[web_search_tool],
    # 使用openrouter接口必须禁用store
    # 禁用store后百炼接口也正常了
    store_enabled=False,
)

In [ ]:
texts_to_search = [
    "Find me news articles about the latest technology trends.",
    "Find me news articles about latest LLM models from different organizations",
]

In [ ]:
thread: AgentThread = None

for user_input in texts_to_search:
    print(f"[red]User ▶︎ {user_input}[/red]")

    response = await agent_ws.get_response(messages=user_input, thread=thread)
    print(f"[green]{response.name} ▶︎ {response.content}[/green]")

    thread = response.thread

# 08 声明式YAML创建agent ✅

In [ ]:
from semantic_kernel.agents import AgentRegistry

In [ ]:
SPEC = """
type: openai_responses
name: Host
instructions: Respond politely to the user's questions.
model:
  id: ${OpenAI:ChatModelId}
tools:
  - id: MenuPlugin.get_specials
    type: function
  - id: MenuPlugin.get_item_price
    type: function
"""

In [ ]:
# user_chat_menu MenuPlugin 从03继承

client = OpenAIResponsesAgent.create_client()

agent: OpenAIResponsesAgent = await AgentRegistry.create_from_yaml(
    SPEC, plugins=[MenuPlugin()], client=client, store_enabled=False
)

In [ ]:
thread: AgentThread = None

for user_input in user_chat_menu:
    print(f"[red]User ▶︎ {user_input}[/red]")

    async for response in agent.invoke(messages=user_input, thread=thread):
        print(f"[green]{response.name} ▶︎ {response.content}[/green]")

        thread = response.thread